In [1]:
# === PARAMETERS ===
MODEL_TYPE = "COORD" # "MLP" or "CNN" or "COORD"
LOG_FILE = "training_log.txt"
NAME_ID = "control_benign"

# Grid / Agents
WIDTH = 6
HEIGHT = 6
NUM_AGENTS = 2
SPAWN_PROB = 0.05
DESPAWN_PROB = 0.09

# MLP Configuration
MLP_HIDDEN_DIM = 512
MLP_NUM_LAYERS = 4

# CNN Configuration (Updated as requested)
CNN_CONV_CHANNELS = [32, 64]
CNN_HEAD_HIDDEN_DIM = 128
CNN_HEAD_NUM_LAYERS = 3
CNN_KERNEL_SIZE = 3

# Training
LR = 0.001
BATCH_SIZE = 64
DISCOUNT = 0.0 # Immediate Reward (Regression)
STOP_THRESHOLD_MAE = 0.01 # 1% Error
SEED = 42
MAX_STEPS = 100000 
EVAL_FREQ = 1000
NUM_TEST_SAMPLES_PER_TYPE = 2000

In [2]:
OUT_FOLDER = f"reward_decentralized_half_{MODEL_TYPE}_lr{LR}_{NAME_ID}"

In [3]:
from pathlib import Path
import sys
import numpy as np
import torch
import random
import ast
from tqdm import tqdm

sys.path.append("../") 

from tadd_helpers.env_functions import init_empty_state, spawn_apples, despawn_apples, State
from tadd_helpers.eval_functions import nearest_policy

# Import Decentralized Models
from models.value_mlp import ValueMLPDecentralized
from models.value_cnn_new import ValueCNNDecentralizedStandard, ValueCNNCoordDecentralized

# --- PARAMETER TYPE SAFETY ---
# If SLURM/Papermill passed a string representation of the list, parse it.
if isinstance(CNN_CONV_CHANNELS, str):
    CNN_CONV_CHANNELS = ast.literal_eval(CNN_CONV_CHANNELS)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

OUT_FOLDER = Path(OUT_FOLDER)
OUT_FOLDER.mkdir(parents=True, exist_ok=True)
LOG_FILE = OUT_FOLDER / LOG_FILE

--- PyTorch is configured to use: cuda ---


In [4]:
# --- INITIALIZATION LOGIC ---
if MODEL_TYPE == "MLP":
    # Use MLP-specific globals
    model = ValueMLPDecentralized(
        height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
        hidden_dim=MLP_HIDDEN_DIM, num_layers=MLP_NUM_LAYERS
    )
    
elif MODEL_TYPE == "CNN":
    # Use CNN-specific globals
    model = ValueCNNDecentralizedStandard(
        height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
        hidden_dim=CNN_HEAD_HIDDEN_DIM, num_layers=CNN_HEAD_NUM_LAYERS, 
        conv_channels=CNN_CONV_CHANNELS, kernel_size=CNN_KERNEL_SIZE
    )
    
elif MODEL_TYPE == "COORD":
    # Use CNN-specific globals (Assuming KERNEL_SIZE=1 was passed in params)
    model = ValueCNNCoordDecentralized(
        height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
        hidden_dim=CNN_HEAD_HIDDEN_DIM, num_layers=CNN_HEAD_NUM_LAYERS, 
        conv_channels=CNN_CONV_CHANNELS, kernel_size=CNN_KERNEL_SIZE
    )
    
else:
    raise ValueError(f"Invalid MODEL_TYPE: {MODEL_TYPE}")

params = sum(p.numel() for p in model.policy_net.parameters())
print(f"Initialized Decentralized {MODEL_TYPE}. Params: {params}. LR: {LR}")

Initialized Decentralized COORD. Params: 348161. LR: 0.001


In [5]:
def generate_decentralized_test_suite(n):
    states_zero = []
    # Store tuples: (State, Picker_ID)
    states_pick = [] 
    
    # 1. Zeros (No agent on ANY apple)
    while len(states_zero) < n:
        s = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
        num_noise = np.random.randint(1, 6)
        cnt = 0; safety = 0
        while cnt < num_noise and safety < 100:
            r, c = np.random.randint(0, HEIGHT), np.random.randint(0, WIDTH)
            if s.agents[r, c] == 0 and s.apples[r, c] == 0:
                s.apples[r, c] = 1; cnt += 1
            safety += 1
        states_zero.append(s)
        
    # 2. Pickers (One agent on apple, others elsewhere)
    while len(states_pick) < n:
        s = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
        s.apples[:] = 0
        
        picker_id = np.random.randint(0, NUM_AGENTS)
        r_t, c_t = np.random.randint(0, HEIGHT), np.random.randint(0, WIDTH)
        
        s.set_agent_position(picker_id, np.array([r_t, c_t]))
        s.apples[r_t, c_t] = 1
        
        # Ensure no overlap
        for other_id in range(NUM_AGENTS):
            if other_id != picker_id:
                while tuple(s.agent_position(other_id)) == (r_t, c_t):
                    r_new, c_new = np.random.randint(0, HEIGHT), np.random.randint(0, WIDTH)
                    s.set_agent_position(other_id, np.array([r_new, c_new]))
        
        # Add noise apples
        num_noise = np.random.randint(0, 5)
        cnt = 0; safety = 0
        while cnt < num_noise and safety < 100:
            r, c = np.random.randint(0, HEIGHT), np.random.randint(0, WIDTH)
            if s.agents[r, c] == 0 and s.apples[r, c] == 0:
                s.apples[r, c] = 1; cnt += 1
            safety += 1
            
        states_pick.append((s, picker_id))
        
    return states_zero, states_pick

test_zeros, test_picks = generate_decentralized_test_suite(NUM_TEST_SAMPLES_PER_TYPE)
print(f"Test Suite: {len(test_zeros)} Zeros, {len(test_picks)} Pickers")

Test Suite: 2000 Zeros, 2000 Pickers


In [6]:
loss_history = []
soft_convergence_step = None
TARGET_REWARD = 1.0 / NUM_AGENTS  # Will be 0.5 for N=2

# Reset Buffer
model.memory.memory.clear()

s: State = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)

with open(LOG_FILE, "a") as f:
    f.write(f"\n=== STARTING DECENTRALIZED CONTROL ({TARGET_REWARD:.2f}): {MODEL_TYPE} [LR={LR}] ===\n")
    f.write("Step   | Loss    | Max%_Z | Avg%_Z | Max%_P | Avg%_P | Max%_O | Avg%_O | Status\n")

pbar = tqdm(range(MAX_STEPS))
for step in pbar:
    
    # 1. MOVE (All agents get 0.0)
    active_agent_idx = s.get_random_agent_id()
    move_action = nearest_policy(s, active_agent_idx)
    new_pos = np.clip(s.agent_position(active_agent_idx) + move_action.vector, [0, 0], [HEIGHT-1, WIDTH-1])
    s_mid = s.copy()
    s_mid.set_agent_position(active_agent_idx, new_pos)
    
    # Store experience for ALL agents
    for i in range(NUM_AGENTS):
        model.add_experience(s, s_mid, 0.0, agent_pos=s.agent_position(i), agent_pos_next=s_mid.agent_position(i))

    # 2. PICK
    s_next = s_mid.copy()
    
    # Check if Active Agent landed on apple
    picker_reward = 0.0
    other_reward = 0.0
    
    if s_next.apples[tuple(new_pos)] > 0:
        s_next.remove_apple_at(new_pos)
        # === CONTROL EXPERIMENT: BENIGN REWARD ===
        picker_reward = TARGET_REWARD # 1/N
        other_reward = TARGET_REWARD  # 1/N
        # =========================================
        
    spawn_apples(s_next, SPAWN_PROB)
    despawn_apples(s_next, DESPAWN_PROB)
    
    # Store Pick experience for ALL agents
    if picker_reward != 0:
        # 1. The Picker
        model.add_experience(s_mid, s_next, picker_reward, agent_pos=s_mid.agent_position(active_agent_idx), agent_pos_next=s_next.agent_position(active_agent_idx))
        # 2. The Others
        for i in range(NUM_AGENTS):
            if i != active_agent_idx:
                model.add_experience(s_mid, s_next, other_reward, agent_pos=s_mid.agent_position(i), agent_pos_next=s_next.agent_position(i))
    else:
        for i in range(NUM_AGENTS):
             model.add_experience(s_mid, s_next, 0.0, agent_pos=s_mid.agent_position(i), agent_pos_next=s_next.agent_position(i))
             
    # 3. TRAIN
    loss = model.train_batch(BATCH_SIZE)
    if loss is not None: loss_history.append(loss)
    s = s_next
    
    # 4. EVAL (Strict 3-Way against TARGET_REWARD)
    if step % EVAL_FREQ == 0 and step > 0:
        
        # A. Zeros (Target 0.0)
        preds_z = np.array([model.get_value(st, agent_pos=st.agent_position(0)) for st in test_zeros])
        if len(preds_z) > 0:
            max_ape_z = np.max(np.abs(preds_z)) * 100
            avg_ape_z = np.mean(np.abs(preds_z)) * 100
        else: max_ape_z, avg_ape_z = 0, 0
        
        # B. Picker Me (Target = TARGET_REWARD)
        preds_pick = np.array([model.get_value(st, agent_pos=st.agent_position(pid)) for st, pid in test_picks])
        if len(preds_pick) > 0:
            max_ape_pick = np.max(np.abs(preds_pick - TARGET_REWARD)) * 100
            avg_ape_pick = np.mean(np.abs(preds_pick - TARGET_REWARD)) * 100
        else: max_ape_pick, avg_ape_pick = 0, 0
        
        # C. Picker Other (Target = TARGET_REWARD)
        preds_oth = []
        for st, pid in test_picks:
            oid = (pid + 1) % NUM_AGENTS
            preds_oth.append(model.get_value(st, agent_pos=st.agent_position(oid)))
        preds_oth = np.array(preds_oth)
        if len(preds_oth) > 0:
            max_ape_oth = np.max(np.abs(preds_oth - TARGET_REWARD)) * 100
            avg_ape_oth = np.mean(np.abs(preds_oth - TARGET_REWARD)) * 100
        else: max_ape_oth, avg_ape_oth = 0, 0
        
        # Global Stats
        global_avg = max(avg_ape_z, avg_ape_pick, avg_ape_oth)
        global_max = max(max_ape_z, max_ape_pick, max_ape_oth) 
        
        status_msg = ""
        if global_avg < 1.0:
            if soft_convergence_step is None:
                soft_convergence_step = step
                print(f"\n🌟 SOFT CONVERGENCE (Avg < 1%) at Step {step}")
            status_msg = "AVG_OK"
            
        if global_max < 1.0:
            status_msg = "STRICT_OK"

        # Log Writing
        with open(LOG_FILE, "a") as f:
            f.write(f"{step:<6} | {loss:.5f} | {max_ape_z:6.2f} | {avg_ape_z:6.2f} | {max_ape_pick:6.2f} | {avg_ape_pick:6.2f} | {max_ape_oth:6.2f} | {avg_ape_oth:6.2f} | {status_msg}\n")
            
        pbar.set_description(f"L:{loss:.4f} | Avg:{global_avg:.1f}% | Max:{global_max:.0f}%")
        
        # STOPPING CONDITION
        if global_max < 1.0:
            print(f"✅ STRICT CONVERGENCE at Step {step}")
            break

  0%|          | 0/100000 [00:00<?, ?it/s]

L:0.0000 | Avg:0.9% | Max:49%:   1%|          | 1041/100000 [00:16<2:18:27, 11.91it/s]


🌟 SOFT CONVERGENCE (Avg < 1%) at Step 1000


L:0.0000 | Avg:0.0% | Max:12%:  36%|███▌      | 36196/100000 [05:40<09:59, 106.39it/s] 


KeyboardInterrupt: 